In [2]:
import re
import numpy as np
import pandas as pd
import requests, os, json
from dotenv import load_dotenv
from collections import defaultdict
from datetime import datetime

In [3]:
load_dotenv()

True

In [4]:
TOKEN = os.getenv("TOKEN")
OWNER_TYPE = "organization"
OWNER_NAME = os.getenv("OWNER_NAME")
PROJECT_NUMBER = int(os.getenv("PROJECT_NUMBER"))

In [5]:
query = """
query($owner: String!, $number: Int!) {
  viewer {
    id
  }
  %s(login: $owner) {
    projectV2(number: $number) {
      items(first: 100) {
        nodes {
          content {
            ... on Issue {
              id
              title
              url
              author {
                login
              }
              createdAt
              updatedAt
              state
              assignees(first: 10) {
                nodes {
                  login
                }
              }
              labels(first: 10) {
                nodes {
                  name
                }
              }
            }
          }
          fieldValues(first: 10) {
            nodes {
              ... on ProjectV2ItemFieldSingleSelectValue {
                name
                field {
                  ... on ProjectV2FieldCommon {
                    name
                  }
                }
              }
            }
          }
        }
      } 
    }
  }
}
""" % OWNER_TYPE

In [6]:
# headers = {"Authorization": f"Bearer {TOKEN}"}
# variables = {"owner": OWNER_NAME, "number": PROJECT_NUMBER}

# # Execute the request
# response = requests.post(
#     "https://api.github.com/graphql",
#     headers=headers,
#     json={"query": query, "variables": variables}
# )


## Parsers

### Bronze Parser

In [7]:
def _get_df(data, status):
  df = pd.DataFrame(data.get(status))
  df = _add_status_to_df(status, df)
  return df

def _add_status_to_df(status: str, df: pd.DataFrame):
  df['status'] = status.lower()
  return df

def transform_raw_to_bronze(data: dict, extracted_at: datetime = None):

    if extracted_at is None:
        extracted_at = datetime.utcnow()

    columns = defaultdict(list)

    items = data.get("data", {}).get(OWNER_TYPE, {}).get("projectV2", {}).get("items", {}).get("nodes", [])

    for item in items:

        if not item.get("content") or "url" not in item["content"]:
            continue

        issue_url = item["content"]["url"]
        issue_id = item["content"]["id"] 
        issue_title = item["content"].get("title", "Untitled")

        author_data = item["content"].get("author")
        username = author_data.get("login") if author_data else "Unknown User"

        created_at = item["content"].get("createdAt", "Unknown")
        updated_at = item["content"].get("updatedAt", "Unknown")
        state = item["content"].get("state", "Unknown")

        assignees_data = item["content"].get("assignees", {}).get("nodes", [])
        assignees = [user.get("login") for user in assignees_data if user]

        labels_data = item["content"].get("labels", {}).get("nodes", [])
        labels_data = [label.get("name") for label in labels_data if label]

        labels = item["content"].get("labels", {}).get("nodes", [])
        
        milestone = None
        if labels:
            for label in labels:
                label = label.get("name")
                if label.startswith("M") and label[1:].isdigit():
                    milestone = label
    
        if milestone == None:
            re.search(r"\[(M\d+)\]", issue_title)        
    
        status = "No Status"
    
        for field in item.get("fieldValues", {}).get("nodes", []):
            if not field:
                continue
            if field.get("field", {}).get("name") == "Status":
                status = field["name"]
                break
    
        columns[status].append({
            "id": issue_id,
            "title": issue_title,
            "url": issue_url,
            "author": username,
            "created_at": created_at,
            "updated_at": updated_at,
            "state": state,
            "assignees": assignees,
            "labels": labels_data,
            "extracted_at": extracted_at,
            "milestone": milestone
        })
    
    ni_df = _get_df(columns, 'Needs Improvement')
    p_df = _get_df(columns, 'Passed')
    ir_df = _get_df(columns, 'In review')
    u_df = _get_df(columns, 'Unchecked/Unsigned')
    
    bronze_table = pd.concat([ni_df, p_df, ir_df, u_df], ignore_index=True)
    bronze_table.rename(
        columns={
            "id": "issue_id"
        }, inplace=True
    )
    
    return bronze_table[bronze_table["title"].str.match(r"\[M\d+\]")]

In [8]:
with open("../data/v2-response.json", 'r') as fp:
    data = json.load(fp)
    print(type(data))

<class 'dict'>


In [9]:
bronze_table = transform_raw_to_bronze(data)
bronze_table.head()

/tmp/ipykernel_47621/1760413641.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  extracted_at = datetime.utcnow()


,issue_id,title,url,author,created_at,updated_at,state,assignees,labels,extracted_at,milestone,status
0,I_kwDOShte388AAAABIWAL1g,[M1] Mark Ivan Contemprato — Weekly Fuel Price...,https://github.com/dataengineeringpilipinas/de...,navikram03,2026-07-10T12:18:57Z,2026-07-30T12:26:36Z,OPEN,[nicholailim],"[milestone-submission, needs-improvement, M1]",2026-08-01 02:49:32.718294,M1,needs improvement
1,I_kwDOShte388AAAABIQvtWQ,[M1] Jerome Fabrero — Philippine Wholesale Ele...,https://github.com/dataengineeringpilipinas/de...,jffabrero,2026-07-09T19:46:39Z,2026-07-29T07:54:07Z,OPEN,[kimodri],"[milestone-submission, M1, ready-for-review]",2026-08-01 02:49:32.718294,M1,needs improvement
2,I_kwDOShte388AAAABH9Hx5w,[M1] supjkay — Beating the Rush: The Metro Man...,https://github.com/dataengineeringpilipinas/de...,supjkay,2026-07-07T13:09:48Z,2026-07-24T14:38:25Z,OPEN,[webzero13],"[milestone-submission, needs-improvement, M1]",2026-08-01 02:49:32.718294,M1,needs improvement
3,I_kwDOShte388AAAABG_LhBA,[M0] Ken Shamrock Dizon — How has bird species...,https://github.com/dataengineeringpilipinas/de...,ksdizon,2026-06-28T22:49:06Z,2026-07-08T17:28:10Z,OPEN,[kimodri],"[milestone-submission, M0]",2026-08-01 02:49:32.718294,M0,needs improvement
4,I_kwDOShte388AAAABGT6ozQ,[M0] Cyan — The AI Divide in Computer Studies ...,https://github.com/dataengineeringpilipinas/de...,penOnFire,2026-06-22T16:17:50Z,2026-07-21T16:39:13Z,OPEN,[webzero13],"[milestone-submission, needs-improvement, M0]",2026-08-01 02:49:32.718294,M0,needs improvement


In [10]:
# specify the type of date
# handle multi-reviewers // explode
# drop labels

silver_table = bronze_table

date_cols = ["created_at", "updated_at", "extracted_at"]

for date_col in date_cols:
    silver_table[date_col] = (
        pd.to_datetime(silver_table[date_col], utc=True)
        .dt.tz_convert("Asia/Manila")
        .dt.floor("s")
    )

silver_table_exploded = silver_table.explode("assignees")\
    .drop(columns=["labels"])



In [11]:
print(len(silver_table_exploded))

83


In [12]:
silver_table_exploded["is_assigned"] = (
    silver_table_exploded["assignees"].notna().astype("int8")
)
silver_table_exploded["days_since_update"] = (
    silver_table_exploded["updated_at"] - silver_table_exploded["created_at"]
).dt.days

silver_table_exploded["submission_age_days"] = (
    silver_table_exploded["extracted_at"] - silver_table_exploded["created_at"]
).dt.days


silver_table_exploded.head()

,issue_id,title,url,author,created_at,updated_at,state,assignees,extracted_at,milestone,status,is_assigned,days_since_update,submission_age_days
0,I_kwDOShte388AAAABIWAL1g,[M1] Mark Ivan Contemprato — Weekly Fuel Price...,https://github.com/dataengineeringpilipinas/de...,navikram03,2026-07-10 20:18:57+08:00,2026-07-30 20:26:36+08:00,OPEN,nicholailim,2026-08-01 10:49:32+08:00,M1,needs improvement,1,20,21
1,I_kwDOShte388AAAABIQvtWQ,[M1] Jerome Fabrero — Philippine Wholesale Ele...,https://github.com/dataengineeringpilipinas/de...,jffabrero,2026-07-10 03:46:39+08:00,2026-07-29 15:54:07+08:00,OPEN,kimodri,2026-08-01 10:49:32+08:00,M1,needs improvement,1,19,22
2,I_kwDOShte388AAAABH9Hx5w,[M1] supjkay — Beating the Rush: The Metro Man...,https://github.com/dataengineeringpilipinas/de...,supjkay,2026-07-07 21:09:48+08:00,2026-07-24 22:38:25+08:00,OPEN,webzero13,2026-08-01 10:49:32+08:00,M1,needs improvement,1,17,24
3,I_kwDOShte388AAAABG_LhBA,[M0] Ken Shamrock Dizon — How has bird species...,https://github.com/dataengineeringpilipinas/de...,ksdizon,2026-06-29 06:49:06+08:00,2026-07-09 01:28:10+08:00,OPEN,kimodri,2026-08-01 10:49:32+08:00,M0,needs improvement,1,9,33
4,I_kwDOShte388AAAABGT6ozQ,[M0] Cyan — The AI Divide in Computer Studies ...,https://github.com/dataengineeringpilipinas/de...,penOnFire,2026-06-23 00:17:50+08:00,2026-07-22 00:39:13+08:00,OPEN,webzero13,2026-08-01 10:49:32+08:00,M0,needs improvement,1,29,39


In [13]:
def _create_dim(
    df: pd.DataFrame,
    natural_key: str,
    surrogate_key: str,
    attributes: list[str] | None = None,
) -> pd.DataFrame:
    columns = [natural_key, *(attributes or [])]
    dim_df = (
        df.loc[df[natural_key].notna(), columns]
        .drop_duplicates(subset=[natural_key])
        .sort_values(natural_key, kind="stable")
        .reset_index(drop=True)
    )
    dim_df.insert(0, surrogate_key, range(1, len(dim_df) + 1))
    return dim_df

In [14]:
DIMENSION_CONFIG = {
    "issue": {
        "natural_key": "issue_id",
        "surrogate_key": "issue_key",
        "attributes": ["title", "url", "author"],
    },
    "reviewer": {
        "natural_key": "assignees",
        "surrogate_key": "reviewer_key",
    },
    "state": {"natural_key": "state", "surrogate_key": "state_key"},
    "status": {"natural_key": "status", "surrogate_key": "status_key"},
    "milestone": {
        "natural_key": "milestone",
        "surrogate_key": "milestone_key",
    },
}

In [15]:
dimensions = {
    name: _create_dim(silver_table_exploded, **config)
    for name, config in DIMENSION_CONFIG.items()
}

issue_attributes = DIMENSION_CONFIG["issue"]["attributes"]
fact_submission_snapshot = silver_table_exploded.drop(
    columns=issue_attributes
).copy()

for name, config in DIMENSION_CONFIG.items():
    natural_key = config["natural_key"]
    surrogate_key = config["surrogate_key"]
    key_mapping = dimensions[name][[surrogate_key, natural_key]]

    fact_submission_snapshot = (
        fact_submission_snapshot.merge(
            key_mapping,
            on=natural_key,
            how="left",
            sort=False,
            validate="many_to_one",
        )
        .drop(columns=natural_key)
    )
    fact_submission_snapshot[surrogate_key] = (
        fact_submission_snapshot[surrogate_key].astype("Int64")
    )


In [24]:
print(type(dimensions))
print(dimensions.keys())

<class 'dict'>
dict_keys(['issue', 'reviewer', 'state', 'status', 'milestone'])


## Assertions

In [17]:
assert len(fact_submission_snapshot) == len(silver_table_exploded)

for name, config in DIMENSION_CONFIG.items():
    natural_key = config["natural_key"]
    surrogate_key = config["surrogate_key"]
    dim_df = dimensions[name]

    assert dim_df[natural_key].notna().all()
    assert dim_df[natural_key].is_unique
    assert dim_df[surrogate_key].tolist() == list(range(1, len(dim_df) + 1))
    pd.testing.assert_frame_equal(
        dim_df, _create_dim(silver_table_exploded, **config)
    )
    source_has_value = silver_table_exploded[natural_key].notna().to_numpy()
    assert fact_submission_snapshot.loc[
        source_has_value, surrogate_key
    ].notna().all()

assert len(dimensions["issue"]) == silver_table_exploded["issue_id"].nunique()
assert not {"title", "url", "author"}.intersection(
    fact_submission_snapshot.columns
)
natural_keys = {config["natural_key"] for config in DIMENSION_CONFIG.values()}
assert not natural_keys.intersection(fact_submission_snapshot.columns)
source_is_unassigned = silver_table_exploded["assignees"].isna().to_numpy()
assert fact_submission_snapshot.loc[
    source_is_unassigned, "reviewer_key"
].isna().all()
assert silver_table_exploded.loc[
    silver_table_exploded["assignees"].isna(), "is_assigned"
].eq(0).all()

In [18]:
fact_submission_snapshot.head()

,created_at,updated_at,extracted_at,is_assigned,days_since_update,submission_age_days,issue_key,reviewer_key,state_key,status_key,milestone_key
0,2026-07-10 20:18:57+08:00,2026-07-30 20:26:36+08:00,2026-08-01 10:49:32+08:00,1,20,21,56,4,2,2,2
1,2026-07-10 03:46:39+08:00,2026-07-29 15:54:07+08:00,2026-08-01 10:49:32+08:00,1,19,22,50,3,2,2,2
2,2026-07-07 21:09:48+08:00,2026-07-24 22:38:25+08:00,2026-08-01 10:49:32+08:00,1,17,24,32,5,2,2,2
3,2026-06-29 06:49:06+08:00,2026-07-09 01:28:10+08:00,2026-08-01 10:49:32+08:00,1,9,33,16,3,2,2,1
4,2026-06-23 00:17:50+08:00,2026-07-22 00:39:13+08:00,2026-08-01 10:49:32+08:00,1,29,39,13,5,2,2,1


In [19]:


def _create_dim_date(df: pd.DataFrame, date_cols: list[str]) -> pd.DataFrame:
    dim_date = (
        pd.concat(
            [df[col] for col in date_cols], ignore_index=True
        )
        .dropna()
        .dt.date
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
        .to_frame(name="date")
    )
    dim_date["date_key"] = dim_date.index + 1
    return dim_date


dim_date = _create_dim_date(fact_submission_snapshot, date_cols)

dim_date.head()

,date,date_key
0,2026-06-22,1
1,2026-06-23,2
2,2026-06-24,3
3,2026-06-25,4
4,2026-06-26,5


In [20]:
fact_submission_snapshot.head()

,created_at,updated_at,extracted_at,is_assigned,days_since_update,submission_age_days,issue_key,reviewer_key,state_key,status_key,milestone_key
0,2026-07-10 20:18:57+08:00,2026-07-30 20:26:36+08:00,2026-08-01 10:49:32+08:00,1,20,21,56,4,2,2,2
1,2026-07-10 03:46:39+08:00,2026-07-29 15:54:07+08:00,2026-08-01 10:49:32+08:00,1,19,22,50,3,2,2,2
2,2026-07-07 21:09:48+08:00,2026-07-24 22:38:25+08:00,2026-08-01 10:49:32+08:00,1,17,24,32,5,2,2,2
3,2026-06-29 06:49:06+08:00,2026-07-09 01:28:10+08:00,2026-08-01 10:49:32+08:00,1,9,33,16,3,2,2,1
4,2026-06-23 00:17:50+08:00,2026-07-22 00:39:13+08:00,2026-08-01 10:49:32+08:00,1,29,39,13,5,2,2,1


In [21]:
def _merge_date_keys_to_fact(
    fact_df: pd.DataFrame, dim_date_df: pd.DataFrame, date_cols: list[str]
) -> pd.DataFrame:
    if not dim_date_df["date"].is_unique:
        raise ValueError("dim_date must contain one row per date")

    fact_df = fact_df.copy()
    date_key_lookup = dim_date_df.set_index("date")["date_key"]

    for col in date_cols:
        fact_df[f"{col}_key"] = (
            fact_df[col]
            .dt.date
            .map(date_key_lookup)
            .astype("Int64")
        )
    return fact_df


fact_row_count = len(fact_submission_snapshot)
fact_submission_snapshot = _merge_date_keys_to_fact(
    fact_submission_snapshot, dim_date, date_cols
)

fact_submission_snapshot.head()

,created_at,updated_at,extracted_at,is_assigned,days_since_update,submission_age_days,issue_key,reviewer_key,state_key,status_key,milestone_key,created_at_key,updated_at_key
0,2026-07-10 20:18:57+08:00,2026-07-30 20:26:36+08:00,2026-08-01 10:49:32+08:00,1,20,21,56,4,2,2,2,15,34
1,2026-07-10 03:46:39+08:00,2026-07-29 15:54:07+08:00,2026-08-01 10:49:32+08:00,1,19,22,50,3,2,2,2,15,33
2,2026-07-07 21:09:48+08:00,2026-07-24 22:38:25+08:00,2026-08-01 10:49:32+08:00,1,17,24,32,5,2,2,2,12,28
3,2026-06-29 06:49:06+08:00,2026-07-09 01:28:10+08:00,2026-08-01 10:49:32+08:00,1,9,33,16,3,2,2,1,8,14
4,2026-06-23 00:17:50+08:00,2026-07-22 00:39:13+08:00,2026-08-01 10:49:32+08:00,1,29,39,13,5,2,2,1,2,26


In [22]:
assert len(fact_submission_snapshot) == fact_row_count

for col in date_cols:
    date_key_col = f"{col}_key"
    source_has_date = fact_submission_snapshot[col].notna()

    assert fact_submission_snapshot[date_key_col].dtype == "Int64"
    assert fact_submission_snapshot.loc[
        source_has_date, date_key_col
    ].notna().all()

## Testing Silver Parsers

In [5]:
import pandas as pd

DATE_COLUMNS = ["created_at", "updated_at", "extracted_at"]

DIMENSION_CONFIG = {
    "issue": {
        "natural_key": "issue_id",
        "surrogate_key": "issue_key",
        "attributes": ["title", "url", "author"],
    },
    "reviewer": {
        "natural_key": "assignees",
        "surrogate_key": "reviewer_key",
    },
    "state": {"natural_key": "state", "surrogate_key": "state_key"},
    "status": {"natural_key": "status", "surrogate_key": "status_key"},
    "milestone": {
        "natural_key": "milestone",
        "surrogate_key": "milestone_key",
    },
}

def _standardize_date_cols(
    table: pd.DataFrame, 
    date_cols: list[str]
)-> pd.DataFrame:
    for date_col in date_cols:
        table[date_col] = (
            pd.to_datetime(table[date_col], utc=True)
            .dt.tz_convert("Asia/Manila")
            .dt.floor("s")
        )
    return table

def _create_dim(
    df: pd.DataFrame,
    natural_key: str,
    surrogate_key: str,
    attributes: list[str] | None = None,
) -> pd.DataFrame:
    columns = [natural_key, *(attributes or [])]
    dim_df = (
        df.loc[df[natural_key].notna(), columns]
        .drop_duplicates(subset=[natural_key])
        .sort_values(natural_key, kind="stable")
        .reset_index(drop=True)
    )
    dim_df.insert(0, surrogate_key, range(1, len(dim_df) + 1))
    return dim_df

def _create_dim_date(df: pd.DataFrame, date_cols: list[str]) -> pd.DataFrame:
    dim_date = (
        pd.concat(
            [df[col] for col in date_cols], ignore_index=True
        )
        .dropna()
        .dt.date
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
        .to_frame(name="date")
    )
    dim_date["date_key"] = dim_date.index + 1
    return dim_date

# def _filter_date_dim(df: pd.DataFrame) -> pd.DataFrame:
#     df["year"] = df["date"].year
#     df["month"] = df["date"].month
#     df["month_name"] = df["date"].month_name()
#     df["day"] = df["date"].day
#     df["day_name"] = df["date"].day_name()

#     return df

def _merge_date_keys_to_fact(
    fact_df: pd.DataFrame, 
    dim_date_df: pd.DataFrame, 
    date_cols: list[str]
) -> pd.DataFrame:
    if not dim_date_df["date"].is_unique:
        raise ValueError("dim_date must contain one row per date")

    fact_df = fact_df.copy()
    date_key_lookup = dim_date_df.set_index("date")["date_key"]

    for col in date_cols:
        fact_df[f"{col}_key"] = (
            fact_df[col]
            .dt.date
            .map(date_key_lookup)
            .astype("Int64")
        )
    return fact_df

# def transform_bronze_to_silver(df: pd.DataFrame) -> dict[str, dict[str, pd.DataFrame]]:
#     silver_table = df.copy()
#     silver_table = _standardize_date_cols(silver_table, DATE_COLUMNS)
#     silver_table_exploded = silver_table.explode("assignees")\
#     .drop(columns=["labels"])

#     # Feature engineer measure columns
#     silver_table_exploded["is_assigned"] = (
#         silver_table_exploded["assignees"].notna().astype("int8")
#     )
#     silver_table_exploded["days_since_update"] = (
#         silver_table_exploded["updated_at"] - silver_table_exploded["created_at"]
#     ).dt.days

#     silver_table_exploded["submission_age_days"] = (
#         silver_table_exploded["extracted_at"] - silver_table_exploded["created_at"]
#     ).dt.days

#     # Create dimensions
#     dimensions = {
#         name: _create_dim(silver_table_exploded, **config)
#         for name, config in DIMENSION_CONFIG.items()
#     }

#     issue_attributes = DIMENSION_CONFIG["issue"]["attributes"]
#     fact_submission_snapshot = silver_table_exploded.drop(
#         columns=issue_attributes
#     ).copy()

#     for name, config in DIMENSION_CONFIG.items():
#         natural_key = config["natural_key"]
#         surrogate_key = config["surrogate_key"]
#         key_mapping = dimensions[name][[surrogate_key, natural_key]]

#         fact_submission_snapshot = (
#             fact_submission_snapshot.merge(
#                 key_mapping,
#                 on=natural_key,
#                 how="left",
#                 sort=False,
#                 validate="many_to_one",
#             )
#             .drop(columns=natural_key)
#         )
#         fact_submission_snapshot[surrogate_key] = (
#             fact_submission_snapshot[surrogate_key].astype("Int64")
#         )

#         # Create date dimension
#         dim_date = _create_dim_date(fact_submission_snapshot, DATE_COLUMNS)
    
#         fact_submission_snapshot = _merge_date_keys_to_fact(
#             fact_submission_snapshot, dim_date, DATE_COLUMNS
#         )
        
#         # dimensions["dim_date"] = _filter_date_dim(dim_date)
        
#         tables = {}
        
#         tables["dimensions"] = dimensions
#         tables["fact_submission_snapshot"] = fact_submission_snapshot

#     return tables

# if __name__ == "__main__":
#     # Example usage
#     bronze_table = pd.read_csv("../data/bronze/bronze_table.csv")
#     silver_tables = transform_bronze_to_silver(bronze_table)
#     print(silver_tables.keys())

In [10]:
df = pd.read_csv("../data/bronze/bronze_table.csv")

df.head()



,issue_id,title,url,author,created_at,updated_at,state,assignees,labels,extracted_at,milestone,status
0,I_kwDOShte388AAAABIWAL1g,[M1] Mark Ivan Contemprato — Weekly Fuel Price...,https://github.com/dataengineeringpilipinas/de...,navikram03,2026-07-10T12:18:57Z,2026-07-30T12:26:36Z,OPEN,['nicholailim'],"['milestone-submission', 'needs-improvement', ...",2026-08-01 03:13:18.520487,M1,needs improvement
1,I_kwDOShte388AAAABIQvtWQ,[M1] Jerome Fabrero — Philippine Wholesale Ele...,https://github.com/dataengineeringpilipinas/de...,jffabrero,2026-07-09T19:46:39Z,2026-07-29T07:54:07Z,OPEN,['kimodri'],"['milestone-submission', 'M1', 'ready-for-revi...",2026-08-01 03:13:18.520487,M1,needs improvement
2,I_kwDOShte388AAAABH9Hx5w,[M1] supjkay — Beating the Rush: The Metro Man...,https://github.com/dataengineeringpilipinas/de...,supjkay,2026-07-07T13:09:48Z,2026-07-24T14:38:25Z,OPEN,['webzero13'],"['milestone-submission', 'needs-improvement', ...",2026-08-01 03:13:18.520487,M1,needs improvement
3,I_kwDOShte388AAAABG_LhBA,[M0] Ken Shamrock Dizon — How has bird species...,https://github.com/dataengineeringpilipinas/de...,ksdizon,2026-06-28T22:49:06Z,2026-07-08T17:28:10Z,OPEN,['kimodri'],"['milestone-submission', 'M0']",2026-08-01 03:13:18.520487,M0,needs improvement
4,I_kwDOShte388AAAABGT6ozQ,[M0] Cyan — The AI Divide in Computer Studies ...,https://github.com/dataengineeringpilipinas/de...,penOnFire,2026-06-22T16:17:50Z,2026-07-21T16:39:13Z,OPEN,['webzero13'],"['milestone-submission', 'needs-improvement', ...",2026-08-01 03:13:18.520487,M0,needs improvement


In [11]:
silver_table = df.copy()
silver_table = _standardize_date_cols(silver_table, DATE_COLUMNS)
silver_table_exploded = silver_table.explode("assignees").drop(columns=["labels"])


In [12]:
silver_table_exploded.head()

,issue_id,title,url,author,created_at,updated_at,state,assignees,extracted_at,milestone,status
0,I_kwDOShte388AAAABIWAL1g,[M1] Mark Ivan Contemprato — Weekly Fuel Price...,https://github.com/dataengineeringpilipinas/de...,navikram03,2026-07-10 20:18:57+08:00,2026-07-30 20:26:36+08:00,OPEN,['nicholailim'],2026-08-01 11:13:18+08:00,M1,needs improvement
1,I_kwDOShte388AAAABIQvtWQ,[M1] Jerome Fabrero — Philippine Wholesale Ele...,https://github.com/dataengineeringpilipinas/de...,jffabrero,2026-07-10 03:46:39+08:00,2026-07-29 15:54:07+08:00,OPEN,['kimodri'],2026-08-01 11:13:18+08:00,M1,needs improvement
2,I_kwDOShte388AAAABH9Hx5w,[M1] supjkay — Beating the Rush: The Metro Man...,https://github.com/dataengineeringpilipinas/de...,supjkay,2026-07-07 21:09:48+08:00,2026-07-24 22:38:25+08:00,OPEN,['webzero13'],2026-08-01 11:13:18+08:00,M1,needs improvement
3,I_kwDOShte388AAAABG_LhBA,[M0] Ken Shamrock Dizon — How has bird species...,https://github.com/dataengineeringpilipinas/de...,ksdizon,2026-06-29 06:49:06+08:00,2026-07-09 01:28:10+08:00,OPEN,['kimodri'],2026-08-01 11:13:18+08:00,M0,needs improvement
4,I_kwDOShte388AAAABGT6ozQ,[M0] Cyan — The AI Divide in Computer Studies ...,https://github.com/dataengineeringpilipinas/de...,penOnFire,2026-06-23 00:17:50+08:00,2026-07-22 00:39:13+08:00,OPEN,['webzero13'],2026-08-01 11:13:18+08:00,M0,needs improvement


In [8]:
# Feature engineer measure columns
silver_table_exploded["is_assigned"] = (
    silver_table_exploded["assignees"].notna().astype("int8")
)
silver_table_exploded["days_since_update"] = (
    silver_table_exploded["updated_at"] - silver_table_exploded["created_at"]
).dt.days

silver_table_exploded["submission_age_days"] = (
    silver_table_exploded["extracted_at"] - silver_table_exploded["created_at"]
).dt.days


silver_table_exploded.head()

,issue_id,title,url,author,created_at,updated_at,state,assignees,extracted_at,milestone,status,is_assigned,days_since_update,submission_age_days
0,I_kwDOShte388AAAABIWAL1g,[M1] Mark Ivan Contemprato — Weekly Fuel Price...,https://github.com/dataengineeringpilipinas/de...,navikram03,2026-07-10 20:18:57+08:00,2026-07-30 20:26:36+08:00,OPEN,['nicholailim'],2026-08-01 11:13:18+08:00,M1,needs improvement,1,20,21
1,I_kwDOShte388AAAABIQvtWQ,[M1] Jerome Fabrero — Philippine Wholesale Ele...,https://github.com/dataengineeringpilipinas/de...,jffabrero,2026-07-10 03:46:39+08:00,2026-07-29 15:54:07+08:00,OPEN,['kimodri'],2026-08-01 11:13:18+08:00,M1,needs improvement,1,19,22
2,I_kwDOShte388AAAABH9Hx5w,[M1] supjkay — Beating the Rush: The Metro Man...,https://github.com/dataengineeringpilipinas/de...,supjkay,2026-07-07 21:09:48+08:00,2026-07-24 22:38:25+08:00,OPEN,['webzero13'],2026-08-01 11:13:18+08:00,M1,needs improvement,1,17,24
3,I_kwDOShte388AAAABG_LhBA,[M0] Ken Shamrock Dizon — How has bird species...,https://github.com/dataengineeringpilipinas/de...,ksdizon,2026-06-29 06:49:06+08:00,2026-07-09 01:28:10+08:00,OPEN,['kimodri'],2026-08-01 11:13:18+08:00,M0,needs improvement,1,9,33
4,I_kwDOShte388AAAABGT6ozQ,[M0] Cyan — The AI Divide in Computer Studies ...,https://github.com/dataengineeringpilipinas/de...,penOnFire,2026-06-23 00:17:50+08:00,2026-07-22 00:39:13+08:00,OPEN,['webzero13'],2026-08-01 11:13:18+08:00,M0,needs improvement,1,29,39


In [ ]:
# Create dimensions
dimensions = {
    name: _create_dim(silver_table_exploded, **config)
    for name, config in DIMENSION_CONFIG.items()
}

issue_attributes = DIMENSION_CONFIG["issue"]["attributes"]
fact_submission_snapshot = silver_table_exploded.drop(
    columns=issue_attributes
).copy()

for name, config in DIMENSION_CONFIG.items():
    natural_key = config["natural_key"]
    surrogate_key = config["surrogate_key"]
    key_mapping = dimensions[name][[surrogate_key, natural_key]]

    fact_submission_snapshot = (
        fact_submission_snapshot.merge(
            key_mapping,
            on=natural_key,
            how="left",
            sort=False,
            validate="many_to_one",
        )
        .drop(columns=natural_key)
    )
    fact_submission_snapshot[surrogate_key] = (
        fact_submission_snapshot[surrogate_key].astype("Int64")
    )

    # Create date dimension
    dim_date = _create_dim_date(fact_submission_snapshot, DATE_COLUMNS)

    fact_submission_snapshot = _merge_date_keys_to_fact(
        fact_submission_snapshot, dim_date, DATE_COLUMNS
    )
    
    # dimensions["dim_date"] = _filter_date_dim(dim_date)
    
    tables = {}
    
    tables["dimensions"] = dimensions
    tables["fact_submission_snapshot"] = fact_submission_snapshot


